In [1]:
from ipyparallel import Cluster

c = await Cluster(engines="mpi").start_and_connect(n=4, activate=True)

Starting 4 engines with <class 'ipyparallel.cluster.launcher.MPIEngineSetLauncher'>


  0%|          | 0/4 [00:00<?, ?engine/s]

In [2]:
%%px
from mpi4py.MPI import COMM_WORLD as comm
from netgen.occ import *
from ngsolve import *
from ngsolve.krylovspace import BramblePasciakCG

box = Box((0, 0, 0), (2, 0.41, 0.41))
box.faces.name = "wall"
box.faces.Min(X).name = "inlet"
box.faces.Max(X).name = "outlet"
cyl = Cylinder((0.2, 0, 0.2), Y, h=0.41, r=0.05)
cyl.faces.name = "cyl"
shape = box - cyl
ngmesh = OCCGeometry(shape).GenerateMesh(maxh=0.05, comm=comm)

for r in range(1):
    ngmesh.Refine()
mesh = Mesh(ngmesh)
print(mesh.GetNE(VOL))

[stdout:0] 0


[stdout:3] 37208


[stdout:2] 35768


[stdout:1] 35760


In [3]:
%%px
import ngsolve.ngs2petsc as n2p
import petsc4py.PETSc as psc

In [4]:
%%px
V = VectorH1(mesh, order=1, dirichlet="wall|inlet|cyl")
V1 = H1(mesh, order=1, dirichlet="wall|inlet|cyl")
Q = H1(mesh, order=1)
printonce("ndof = ", V.ndofglobal, "+", Q.ndofglobal, "=", V.ndofglobal + Q.ndofglobal)

u, v = V.TnT()
u1, v1 = V1.TnT()
p, q = Q.TnT()

h = specialcf.mesh_size

bfa1 = BilinearForm(InnerProduct(grad(u1), grad(v1)) * dx)
bfb = BilinearForm(div(u) * q * dx).Assemble()
bfc = BilinearForm(h * h * grad(p) * grad(q) * dx).Assemble()

prea1 = Preconditioner(bfa1, "gamg")  # AMG precond from PETSc
bfa1.Assemble()

# make block-diagonal A matrix:
mata = sum([Ri.T @ bfa1.mat @ Ri for Ri in V.restrictions])
prea = sum([Ei @ prea1 @ Ei.T for Ei in V.embeddings])

bfschur = BilinearForm(p * q * dx, diagonal=True).Assemble()
preschur = bfschur.mat.Inverse()

[stdout:0] ndof =  66090 + 22030 = 88120


In [5]:
%%px
gfu = GridFunction(V)
gfp = GridFunction(Q)

uin = (1.5 * 4 * y * (0.41 - y) / (0.41 * 0.41) * z * (0.41 - z) / 0.41**2, 0, 0)

gfu.Set(uin, definedon=mesh.Boundaries("inlet"))

resf = (-mata * gfu.vec).Evaluate()
resg = (-bfb.mat * gfu.vec).Evaluate()

sol = BramblePasciakCG(
    A=mata,
    B=bfb.mat,
    C=bfc.mat,
    f=resf,
    g=resg,
    preA=prea,
    preS=preschur,
    maxit=500,
    printrates="\r" if comm.rank == 0 else False,
)

gfu.vec.data += sol[0]
gfp.vec.data += sol[1]

[stdout:0] lammin/lammax =  0.5052418810348174 / 0.9982329720510617


In [6]:
gfu = c[:]["gfu"]

In [7]:
from ngsolve import *
from ngsolve.webgui import Draw

ea = {"euler_angles": (-77, 6, 47)}
clipping = {"clipping": {"y": 1, "z": 0, "function": True}}
Draw(Norm(gfu[0]), gfu[0].space.mesh, **ea, **clipping, order=1)
Draw(Norm(gfu[2]), gfu[2].space.mesh, **ea, **clipping, order=1);

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {'camera': {'euler_angles': (-…

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {'camera': {'euler_angles': (-…

In [8]:
gfp = c[:]["gfp"]
Draw(gfp[0], order=1, **ea, **clipping);

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {'camera': {'euler_angles': (-…

In [9]:
c.shutdown(hub=True)

Output for ipengine-1776418834-t83w-1776418835-974533:
2026-04-17 16:40:39.037 [IPEngine.1.1] Connecting shell to tcp://127.0.0.1:37887
2026-04-17 16:40:39.037 [IPEngine.1.1] Connecting shell to tcp://127.0.0.1:33227
2026-04-17 16:40:39.037 [IPEngine.1.1] Starting nanny
2026-04-17 16:40:39.037 [IPEngine.3.3] Shell_addrs: ['tcp://127.0.0.1:55447', 'tcp://127.0.0.1:37887', 'tcp://127.0.0.1:33227']
2026-04-17 16:40:39.037 [IPEngine.3.3] Connecting shell to tcp://127.0.0.1:55447
2026-04-17 16:40:39.037 [IPEngine.3.3] Connecting shell to tcp://127.0.0.1:37887
2026-04-17 16:40:39.037 [IPEngine.3.3] Connecting shell to tcp://127.0.0.1:33227
2026-04-17 16:40:39.037 [IPEngine.3.3] Starting nanny
2026-04-17 16:40:39.037 [IPEngine.2.2] Shell_addrs: ['tcp://127.0.0.1:55447', 'tcp://127.0.0.1:37887', 'tcp://127.0.0.1:58425']
2026-04-17 16:40:39.037 [IPEngine.2.2] Connecting shell to tcp://127.0.0.1:55447
2026-04-17 16:40:39.037 [IPEngine.2.2] Connecting shell to tcp://127.0.0.1:37887
2026-04-17 16: